# Lesson 7 — Assemble & train the GPT

Every piece so far, in order: token + position embeddings (L3) → a stack of blocks (L6, each doing masked
attention L4/L5 + feed-forward) → final layer norm → `lm_head` that turns each position into 65 logits, one per
character. With targets it returns the cross-entropy loss from L1. Then I train it.

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(1337)

## Data: encode the text, 90% train / 10% validation

In [2]:
text = open("shakespeare.txt").read()
chars = sorted(set(text)); vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]

## Config — small enough for a CPU
Context of 32 characters, 64-dim embeddings, 4 heads, 3 blocks.

In [3]:
# --- model config (small, so it trains on a CPU in a couple of minutes) ---
block_size = 32     # context length
n_embd     = 64     # embedding size
n_heads    = 4      # attention heads
n_layers   = 3      # transformer blocks
batch_size = 16

## The building blocks from lesson 6 (unchanged)

In [4]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
        self.proj  = nn.Linear(n_heads * head_size, n_embd)
    def forward(self, x):
        return self.proj(torch.cat([h(x) for h in self.heads], dim=-1))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.ReLU(), nn.Linear(4 * n_embd, n_embd))
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_heads):
        super().__init__()
        self.sa  = MultiHeadAttention(n_heads, n_embd // n_heads)
        self.ff  = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd); self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

## The GPT
- `forward`: embed → blocks → `ln_f` → `lm_head` logits `(B, T, vocab)`; with targets, flatten and take cross-entropy.
- `generate`: repeatedly feed the last `block_size` tokens, softmax the last position's logits, sample one character,
  append it.

In [5]:
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks  = nn.Sequential(*[Block(n_embd, n_heads) for _ in range(n_layers)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding(idx) + self.position_embedding(torch.arange(T))
        x = self.ln_f(self.blocks(x))
        logits = self.lm_head(x)                       # (B, T, vocab)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

In [6]:
model = GPT()
print("total parameters:", sum(p.numel() for p in model.parameters()))

total parameters: 159937


## Sanity check before training
An untrained model should be about as good as guessing uniformly, so the loss should sit near `log(65)`.

Note: the course code below calls `torch.randint` **twice**, so `xb` and `yb` come from *different* random positions
and the targets don't line up with the inputs. I kept it exactly as the course has it. It doesn't matter here,
because an untrained model is just guessing either way.

In [7]:
# one forward pass before any training — loss should sit near log(vocab_size)
xb = torch.stack([train_data[i:i + block_size] for i in torch.randint(len(train_data) - block_size, (batch_size,))])
yb = torch.stack([train_data[i + 1:i + block_size + 1] for i in torch.randint(len(train_data) - block_size, (batch_size,))])
_, loss = model(xb, yb)
import math
print(f"untrained loss: {loss.item():.3f}   (random guessing = log({vocab_size}) = {math.log(vocab_size):.3f})")

untrained loss: 4.450   (random guessing = log(65) = 4.174)


## Training loop
Each step: get a batch → compute loss → backprop → AdamW update. Every 100 steps `estimate_loss` averages 10
batches of train and of val (in eval mode, no gradients) so I get a less noisy reading.

In [8]:
def get_batch(split):
    src = train_data if split == "train" else val_data
    ix = torch.randint(len(src) - block_size, (batch_size,))
    x = torch.stack([src[i:i + block_size] for i in ix])
    y = torch.stack([src[i + 1:i + block_size + 1] for i in ix])
    return x, y

@torch.no_grad()
def estimate_loss():
    model.eval(); out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(10)
        for k in range(10):
            _, l = model(*get_batch(split)); losses[k] = l.item()
        out[split] = losses.mean().item()
    model.train(); return out

In [9]:
import time; t0 = time.time()
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
max_iters, eval_interval = 400, 100
for it in range(max_iters + 1):
    if it % eval_interval == 0:
        l = estimate_loss(); print(f"step {it:4d}:  train loss {l['train']:.3f}   val loss {l['val']:.3f}")
    _, loss = model(*get_batch("train"))
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()

torch.save(model.state_dict(), "gpt.pt")     # save for Lessons 8-9
print("\nsaved gpt.pt")
print(f"(training took {time.time() - t0:.1f}s on CPU)")

step    0:  train loss 4.409   val loss 4.409


step  100:  train loss 2.518   val loss 2.517


step  200:  train loss 2.366   val loss 2.366


step  300:  train loss 2.273   val loss 2.288


step  400:  train loss 2.210   val loss 2.177

saved gpt.pt
(training took 5.9s on CPU)


## Let it write
Start from token 0 (a newline) and sample 400 characters.

In [10]:
context = torch.zeros((1, 1), dtype=torch.long)     # a single start token
sample_text = decode(model.generate(context, max_new_tokens=400)[0].tolist())
print(sample_text)


A knes, en myours Warmalior sood'ss
qomsurmame whows, af my bumessir man and you dence;
le eferee amerp sould bins alt o sis morcen?
Whou thydive ent soverectere not bet mat an
Whet ou, be the to it theraurr, maged:
Tise, mmes lord cose diviongs lond!
Heldo
TAnd's wore, lamiventautourayge pueseaws my an erie onse he enge to oldefue.
wes shas of cik boir wo bled ast-oner, thae,
We basethes Ghou thi


## Reference check against the course output

In [11]:
course = {"params": 159937, "untrained": 4.450,
          "curve": [(0, 4.409, 4.409), (100, 2.518, 2.517), (200, 2.366, 2.367), (300, 2.275, 2.287), (400, 2.209, 2.177)]}
print("parameters   course", course["params"], "  mine", sum(p.numel() for p in model.parameters()),
      "  match:", course["params"] == sum(p.numel() for p in model.parameters()))
final = estimate_loss()   # fresh (random) batches after training, just to confirm the level
print(f"final loss re-estimated now: train {final['train']:.3f}  val {final['val']:.3f}  (course step 400: 2.209 / 2.177)")
lines = [ln for ln in sample_text.splitlines() if ln.strip()]
print("sample has line breaks          :", len(lines), "non-empty lines")
print("lines starting with a capital   :", sum(ln[0].isupper() for ln in lines), "of", len(lines))
print("colon-tagged words (Name: style):", [w for w in sample_text.split() if w.endswith(":")])
course_start = "\nA knes, en myours Warmaliov:"
same = next((i for i, (a, b) in enumerate(zip(sample_text, course_start)) if a != b), len(course_start))
print(f"sample identical to the course for the first {same} characters, then drifts")

parameters   course 159937   mine 159937   match: True
final loss re-estimated now: train 2.198  val 2.206  (course step 400: 2.209 / 2.177)
sample has line breaks          : 10 non-empty lines
lines starting with a capital   : 7 of 10
colon-tagged words (Name: style): ['maged:']
sample identical to the course for the first 27 characters, then drifts


**Exact matches:** 159937 parameters, untrained loss 4.450, log(65) = 4.174.

**Seeded numbers differ very slightly** (torch 2.15.0 here). The loss curve, course vs mine:

| step | course train | mine train | course val | mine val |
|-----:|-----:|-----:|-----:|-----:|
| 0   | 4.409 | 4.409 | 4.409 | 4.409 |
| 100 | 2.518 | 2.518 | 2.517 | 2.517 |
| 200 | 2.366 | 2.366 | 2.367 | **2.366** |
| 300 | 2.275 | **2.273** | 2.287 | **2.288** |
| 400 | 2.209 | **2.210** | 2.177 | 2.177 |

The largest gap is 0.002, which comes from floating-point rounding in a different PyTorch build. Because the
weights are a hair different, the sample matches the course's for its first 27 characters
(`A knes, en myours Warmalio…`) and then goes its own way, since every sampled character changes what comes next.

**Behaviour is the same:** loss starts ≈4.4 and falls to ≈2.2, train and val track each other, and the sample has
line breaks, mostly capitalised line starts (7 of 10), punctuation and word-shaped chunks. The course's sample has
one `word:` tag (`Warmaliov:`) and mine has one (`maged:`). Neither has a clean `NAME:` speaker line in these 400
characters. (The re-estimated loss after training uses fresh random batches, so it wobbles a little around 2.2.)

## Test yourself

**1. Why should the untrained loss be close to log(65) ≈ 4.17?**
A fresh model gives roughly equal probability to all 65 characters, so the cross-entropy is −log(1/65) = log 65.
Being a bit above it (4.45) just means the random initial guesses aren't perfectly flat.

**2. What does it tell you that train and val loss stay close?**
The model is learning patterns that carry over to text it hasn't trained on, not memorising the training set.

**3. In `generate`, why crop the input to `idx[:, -block_size:]`?**
The position embedding only has 32 rows, so the model can't take more than 32 tokens. Once the text is longer, I
feed it the most recent 32.

## My notes
- A GPT is just embeddings → stacked blocks → a linear layer to vocab logits; training is the same 4-step loop as any net.
- Surprising: with only ~160k parameters and 400 steps it already learned line breaks, capitals and `NAME:` speaker tags.